In [35]:
import pandas as pd
import pyodbc
import numpy as np

In [ ]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=127.0.0.1,1433;"
    "DATABASE=EcomRetentionDB;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)
df = pd.read_sql("SELECT COUNT(*) AS order_count FROM mart.vw_order_revenue", conn)
print(df)

In [ ]:
customers = pd.read_sql("SELECT * FROM clean.customers",conn)
revenue = pd.read_sql("SELECT * FROM mart.vw_order_revenue",conn)
print("customer Rows:",len(customers))
print("revenue Rows:",len(revenue))

In [4]:
customers.head(5)

,customer_id,customer_name,city,state,pincode,signup_date
0,CUST00001,Lakshmi Thakur,Pune,Maharashtra,411698,26-11-2022
1,CUST00002,Meera Joshi,Kolkata,West Bengal,700660,22-03-2023
2,CUST00003,Vihaan Chopra,Ahmedabad,Gujarat,380913,16-03-2021
3,CUST00004,Manish Singh,Delhi,Delhi,110365,10-08-2021
4,CUST00005,Deepa Agarwal,Delhi,Delhi,110754,31-03-2024


In [5]:
customers.dtypes

customer_id      str
customer_name    str
city             str
state            str
pincode          str
signup_date      str
dtype: object

In [6]:
revenue.dtypes

order_id             str
customer_id          str
order_date        object
order_status         str
channel              str
order_revenue    float64
dtype: object

In [7]:
customers.duplicated('customer_id').sum()

np.int64(0)

In [8]:
customers.isna().sum()

customer_id      0
customer_name    0
city             0
state            0
pincode          0
signup_date      0
dtype: int64

In [9]:
revenue.isna().sum()

order_id           0
customer_id        0
order_date         0
order_status       0
channel            0
order_revenue    129
dtype: int64

In [10]:
revenue['order_date'] = pd.to_datetime(revenue['order_date'])
print(revenue['order_date'].dtypes)

datetime64[s]


In [11]:
revenue['order_revenue'].describe()

count    17834.000000
mean      5751.293549
std       4489.650642
min        147.000000
25%       2544.275000
50%       4480.250000
75%       8011.150000
max      36330.200000
Name: order_revenue, dtype: float64

In [12]:
# Count the nulls in Python
no_revenue = revenue[revenue['order_revenue'].isna()]
print(len(no_revenue))
no_revenue[['order_id','customer_id','order_date']].head()

129


,order_id,customer_id,order_date
12,ORD003912,CUST02682,2023-03-29
124,ORD016513,CUST05334,2024-04-24
137,ORD017029,CUST04854,2024-09-06
227,ORD004772,CUST00845,2023-11-30
289,ORD012887,CUST04357,2024-07-24


In [ ]:
# Confirm the reason in SSMS
a = pd.read_sql("""SELECT COUNT(*) AS delivered_no_items
FROM clean.orders o
WHERE o.order_status = 'delivered'
  AND NOT EXISTS (
      SELECT 1 FROM clean.order_items i
      WHERE i.order_id = o.order_id
  )""",conn)
print(a)

In [14]:
# Decide the rule
revenue['order_revenue']= revenue['order_revenue'].fillna(0)
print("nulls_left",revenue['order_revenue'].isna().sum())
print("total_orders",len(revenue))

nulls_left 0
total_orders 17963


In [17]:
# Build the RFM base table 
rfm = revenue.groupby('customer_id').agg(
    last_order_date = ('order_date','max'),
    frequency = ('order_id','count'),
    monetary = ('order_revenue','sum')
).reset_index()
snapshot = pd.Timestamp('2025-01-01')
rfm['recency'] = (snapshot - rfm['last_order_date']).dt.days
rfm = customers[['customer_id']].merge(rfm, on='customer_id',how='left')
rfm['frequency'] = rfm['frequency'].fillna(0).astype(int)
rfm['monetary'] = rfm['monetary'].fillna(0)
print("rows:",len(rfm))
print("Customers with frequency 0:",(rfm['frequency']==0).sum())
rfm.head(2)

rows: 8000
Customers with frequency 0: 2230


,customer_id,last_order_date,frequency,monetary,recency
0,CUST00001,2024-12-23,2,3545.55,9.0
1,CUST00002,2024-06-25,3,23523.70,190.0


In [28]:
#Score each customer 1 to 5 on R, F, and M 
active = rfm[rfm['frequency']>0].copy() 
active['r_score'] = pd.qcut(active['recency'].rank(method = 'first',ascending = False), 5 , labels = [1,2,3,4,5]).astype(int) 
active['f_score'] = pd.qcut(active['frequency'].rank(method = 'first'), 5, labels = [1,2,3,4,5]).astype(int)
active['m_score'] = pd.qcut(active['monetary'].rank(method = 'first'),5,labels = [1,2,3,4,5]).astype(int) 
rfm = rfm.merge(active[['customer_id','r_score','f_score','m_score']],on = 'customer_id',how = 'left') 
for col in ['r_score','f_score','m_score']: rfm[col] = rfm[col].fillna(1).astype(int)
print("r_score(active_customer):") 
print(active['r_score'].value_counts().sort_index()) 
print("\nf_score(active_customers):") 
print(active['f_score'].value_counts().sort_index()) 
print("\nm_score (active customers):") 
print(active['m_score'].value_counts().sort_index())

r_score(active_customer):
r_score
1    1154
2    1154
3    1154
4    1154
5    1154
Name: count, dtype: int64

f_score(active_customers):
f_score
1    1154
2    1154
3    1154
4    1154
5    1154
Name: count, dtype: int64

m_score (active customers):
m_score
1    1154
2    1154
3    1154
4    1154
5    1154
Name: count, dtype: int64


In [34]:
# Assign segments
rfm['f_score'] = rfm['frequency'].clip(lower = 1, upper = 5).astype(int)
def segment(row):
    if row['frequency']==0:
        return 'No Delivered orders'
    if row['r_score']>= 4 and row['f_score']>=4 and row['m_score']>= 4:
        return 'Champions'
    if row['f_score']>= 3 and row['r_score']>= 3:
        return 'Loyal'
    if row['r_score']>= 4 and row['f_score']== 1:
        return 'New'
    if row['r_score']<= 2 and row['f_score']>=3 :
        return 'At Risk'
    if row['r_score']<=2 and row['f_score']<= 2:
        return 'Lost'
    return 'Needs Attention'
    
rfm['segment'] = rfm.apply(segment,axis=1)
print(rfm['segment'].value_counts())
print("total:",rfm['segment'].value_counts().sum())

segment
No Delivered orders    2230
Lost                   1991
Needs Attention        1374
Loyal                   781
Champions               776
New                     531
At Risk                 317
Name: count, dtype: int64
total: 8000


In [39]:
# Churn flag
rfm['churn'] = np.where((rfm['frequency']==0) | (rfm['recency']>90),1,0)
print(rfm['churn'].value_counts().sort_index())
print("total",rfm['churn'].value_counts().sum())

churn
0    1799
1    6201
Name: count, dtype: int64
total 8000


In [ ]:
# Delivery and review analysis
deli = pd.read_sql("SELECT * FROM mart.vw_delivery_performance",conn)
revi = pd.read_sql("SELECT * FROM clean.reviews",conn)
deli = deli[deli['delivered_date'].notna()].copy()
deli['delay_band']= pd.cut(deli['delay_days'],
                          bins = [-999,0,2,999],
                          labels = ['On time or early','1 or 2 days late','3+ days late'])
print("On time rate by courier:")
print(deli.groupby('courier')['on_time_flag'].mean().round(3))
merged = deli.merge(revi[['order_id','review_score']],on = 'order_id',how = 'inner')
print("\nAverage review score by delay band :")
print(merged.groupby('delay_band',observed = True)['review_score'].agg(['mean','count']).round(2))

In [46]:
#Export the RFM table and run final checks
cols = ['customer_id','last_order_date','frequency','monetary','recency','r_score','f_score','m_score','segment','churn']
rfm_out = rfm[cols]

In [47]:
print(rfm_out.isna().sum())

customer_id           0
last_order_date    2230
frequency             0
monetary              0
recency            2230
r_score               0
f_score               0
m_score               0
segment               0
churn                 0
dtype: int64


In [49]:
rfm_out.to_csv('rfm_customers.csv',index=False,encoding = 'utf-8')
print("saved")

saved


In [ ]:
check = pd.read_csv('rfm_customers.csv')
print("rows:", len(check))
print("columns:", list(check.columns))
print("blanks in segment:", check['segment'].isna().sum())
print("blanks in churn:", check['churn'].isna().sum())

## Phase 4 notes: RFM segmentation and churn

**Snapshot date:** 1 January 2025

**Revenue:** delivered orders only, from `mart.vw_order_revenue`. Orders with no valid items have revenue set to 0 (129 orders).

**Churn rule:** a customer is churned (1) if they have no delivered order in the 90 days before the snapshot, or no delivered order at all. Otherwise churn is 0.

**Segment rules** (applied in this order):
1. No delivered orders: customers with frequency 0 (2,230)
2. Champions: R, F, and M scores all 4 or higher
3. Loyal: F and R scores both 3 or higher
4. New: R score 4 or higher, and frequency of 1
5. At Risk: R score 2 or lower, and F score 3 or higher
6. Lost: R score 2 or lower, and F score 2 or lower
7. Needs Attention: everyone else

**Scoring:** R, F, and M are scored 1 to 5 using quintiles. Recency is reversed, so the most recent customers score highest. Ties in frequency are broken by rank, so each group has about 1,154 customers.

**Findings**
- Average review score falls as delivery gets later: 4.48 when on time or early, 3.78 for 1 to 2 days late, 2.47 for 3 or more days late.
- On-time rates by courier are similar, between 44% and 47%.
- About 78% of customers are churned under the 90-day rule. The data covers two years, so most customers have not bought recently at the snapshot date.

**Data note:** Churn counts differ from the generator's files by 28 customers. The likely cause is the Excel re-save of mixed date formats, which changed a few last-order dates.

**Output:** `rfm_customers.csv` (8,000 rows, 10 columns), used by the Power BI dashboard.